# Physarum Dynamic Routing Simulation

This model simulates Physarum finding escape routes in an evironment affected by wildfires and compares this with the traditional routing algorithm: Dijkstra's algorithm. It uses the osmnx python library to download a graph of a real  forest to accurately represent traveling roads in the model. <br>
<br>
The wildfire simulation algorithm in this model is based after: <br>
> "A cellular automata model for forest fire spread prediction: The case of the wildfire that swept through Spetses Island in 1990". Alexandridis, A. et al. https://doi.org/10.1016/j.amc.2008.06.046 <br>
> and <br>
> "Wildland fire spread modelling using cellular automata: evolution in large-scale spatially heterogeneous environments under fire suppression tactics". Alexandridis, A. et al.https://doi.org/10.1071/wf09119

The Physarum algorithm used in this model is based after: <br>
> An Improved Physarum polycephalum Algorithm for the Shortest Path Problem. Zhang, X. et al. https://doi.org/10.1155/2014/487069

The Dijkstra algorithm used in this model is based after: <br>
> A note on two problems in connexion with graphs. Dijkstra, E. et al. https://doi.org/10.1007/bf01386390



### Packages
These are all the packages needed to run the simulation.

In [ ]:
import osmnx as ox
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import networkx as nx 
import time
import heapq

from pyproj import Transformer
from scipy.ndimage import distance_transform_edt, binary_closing
from scipy.ndimage import binary_dilation

## Model parameters
In this sections all model parameters are defined. These can be changed to run different tests. <br>
When changing the model parameters for tests make sure to run all cells below again.

In [ ]:
"""Parameters for downloading the map and basic parameters for the setup of the cells in the simulation"""
#Choose coordinates for the location from which the graph should be made
Map_coordinates = bbox=(2.55, 48.35, 2.67, 48.41)    # Choosen area = Fontainebleau Forest, France

# Cell size is set, this determines the metres per grid cell.
CELL_SIZE_M = 10   # A smaller value makes the grid more detailed but also slower. 
MAX_GRID = 500     # A max grid size is set to make sure the simulation doesn't slow down to much.

#Road sizes for different road types are set. "Bigger" roads show up larger.
ROAD_WIDTHS = {
    "motorway"      : 3,
    "trunk"         : 3,
    "primary"       : 2,
    "secondary"     : 2,
    "tertiary"      : 1,
    "residential"   : 1,
    "unclassified"  : 1,
    "track"         : 1,
    "path"          : 1,
    "service"       : 1,
}
DEFAULT_WIDTH = 1    # If a road type is not found in the Dict then the thickness is set to 1 by default.
EXPAND_CELLS = 1.1   # Roads are widened a bit for visibility and and to make sure there are no gaps in the roads due to unperfect rasterization.



"""Parameters that determine how many exits and villages are placed and how big they are."""
N_EXITS    = 5
EXIT_RADIUS = 10   
VILLAGE_RADIUS = 4

"""A random seed is chosen to make sure one experiment can be run multiple times with different parameters."""
RANDOM_SEED = 60  
sim_rng = np.random.default_rng(RANDOM_SEED)



"""Wildfire parameters"""
N_FIRE_STARTS     = 8       # Determines the amount of fire embers that will be placed. More starts would mean the algorithms would have less room to find paths.
FIRE_SPREAD_PROB  = 0.3     # A higher spreading probability means the wildfire will spread quicker.
HIGH_RISK_RADIUS  = 6       # Determines how large (how many cells in width) the risk area will be.
BURN_DURATION     = 20      # Ignighted fire stays burning for this amount of steps, after that it is burned.
FIRE_SPREAD_EVERY = 2       # Fire only spreads after each FIRE_SPREAD_EVERY timestep. A higher value means the fire spreads slower and the algorithms have more time to find new routes until the fire spreads again.

"""Physarum parameters"""
# I0 is the source/sink parameter. This determines how much physarum is "flowing" from the souce (the village).
# I0 is the lenght of the edges_arr here to make sure the flux is somewhat proportional to the size of the graph. 
# Later in the code the I0 keeps being updated to be proportional to the amount of edges that are still safe: b[node_idx[village_node]] = 1.0 * active_road_count.
# If the I0 would be to small, not enough flux (Q) can be generated to make the Physarum grow faster than it decays. Also if I0 is to low it will never start the positive feedbak loop.
# If the I0 would be to large, all flux will become very high and the algorithm will fail to prune away the long, inefficient routes. This will make it very unclear what the actual shortes route is and just show a lot of alternative routes.
"""I0 = len(edges_arr)"""   # Search for this parameter in the Initialization section if you want to change this. 

threshold_draw = 0.5          # The conductivity of the Physarum must surpass this threshold to show up in the display. If the threshold is larger the Physarum must be "more sure" about its path to show up on the display.
Conductivity_threshold = 0.5  # The conductivity of the Physarum must surpass this threshold to be counted as an existing road in the network. 

""" simulation parameters """
TOTAL_STEPS             = 1200                            # This is the maximum amount of steps the simulation will do. If the village is burned earlier the simulation will stop at that point.
SNAPSHOT_AT             = [10, 50, 100, 175, 225, 300]    # These are the steps at which a snapshot is taken. Play around with these numbers to capture the most interesting snapshots.


### Downloading the map
First the graph must be downloaded with the OSMnx library. 

In [ ]:
ox.settings.timeout = 60  
ox.settings.overpass_rate_limit = False   # skip rate limit checks

ox.settings.use_cache = True
ox.settings.log_console = True 

G = ox.graph_from_bbox(bbox=(2.55, 48.35, 2.67, 48.41), network_type="all", simplify=True)

print(f"Road graph is downloaded")
print(f"Nodes : {len(G.nodes)}")
print(f"Edges : {len(G.edges)}")

### Prepare graph for rasterization
Cell_zize_M is 10.<br>
MAX_GRID is 500. <br>
Here the cell size could still be adjusted to make sure it fits in the max grid. 

In [ ]:
# Project to flat coordinate system from latitude degrees to meters.
G_proj = ox.project_graph(G) 

# Get the boundaries in meters
nodes_proj, edges_proj = ox.graph_to_gdfs(G_proj)
minx = nodes_proj.geometry.x.min()
maxx = nodes_proj.geometry.x.max()
miny = nodes_proj.geometry.y.min()
maxy = nodes_proj.geometry.y.max()

width_m  = maxx - minx
height_m = maxy - miny

print(f"Size of area : {width_m/1000:.1f} km × {height_m/1000:.1f} km")


GRID_COLS = int(width_m  / CELL_SIZE_M)
GRID_ROWS = int(height_m / CELL_SIZE_M)


if GRID_COLS > MAX_GRID or GRID_ROWS > MAX_GRID:
    scale    = MAX_GRID / max(GRID_COLS, GRID_ROWS)
    GRID_COLS = int(GRID_COLS * scale)
    GRID_ROWS = int(GRID_ROWS * scale)
    CELL_SIZE_M = width_m / GRID_COLS
    print(f"Cell sizes are adjusted to {CELL_SIZE_M:.0f} m")

print(f"Grid size : {GRID_ROWS} rows × {GRID_COLS} cols")
print(f"Cell size : {CELL_SIZE_M:.0f} m")

In [ ]:
"""Make sure the graph is undirected for future purposes."""
G_simple = ox.convert.to_undirected(G_proj) 

### Rasterize the OSMnx roads onto the created grid
Here the grid is build and the graph is rasterized ontop of it as the road network. <br>
Roads are widened for visibility and to make sure there are no gaps in the road.

In [ ]:
# mark the graph edges as ROAD on grid cells.
EMPTY   = 0
ROAD    = 1
EXIT    = 2
VILLAGE = 3


def coord_to_grid(x, y):
    """Convert meter coordinates to (row, col)."""
    col = int((x - minx) / CELL_SIZE_M)
    row = int((maxy - y) / CELL_SIZE_M) 
    col = np.clip(col, 0, GRID_COLS - 1)
    row = np.clip(row, 0, GRID_ROWS - 1)
    return row, col


def stamp_cell(grid, row, col, half_width):
    """Stamp roads onto grid"""
    hw = max(0, half_width)
    for dr in range(-hw, hw + 1):
        for dc in range(-hw, hw + 1):
            if dr*dr + dc*dc <= hw*hw + hw:
                nr = np.clip(row + dr, 0, GRID_ROWS - 1)
                nc = np.clip(col + dc, 0, GRID_COLS - 1)
                grid[nr, nc] = ROAD


def rasterize_edge(grid, geom, half_width, n_samples=None):
    
    coords = list(geom.coords)
    for i in range(len(coords) - 1):
        x0, y0 = coords[i]
        x1, y1 = coords[i + 1]
        seg_len = np.hypot(x1 - x0, y1 - y0)
        n       = max(2, int(seg_len / (CELL_SIZE_M * 0.5)))
        for t in np.linspace(0, 1, n):
            x = x0 + t * (x1 - x0)
            y = y0 + t * (y1 - y0)
            r, c = coord_to_grid(x, y)
            stamp_cell(grid, r, c, half_width)


# Build the grid
road_grid = np.zeros((GRID_ROWS, GRID_COLS), dtype=np.int32)
GRID_SIZE = max(GRID_ROWS, GRID_COLS)  

for _, edge in edges_proj.iterrows():
    highway = edge.get("highway", "unclassified")
    if isinstance(highway, list):
        highway = highway[0]   
    hw = ROAD_WIDTHS.get(highway, DEFAULT_WIDTH) - 1   # half_width
    rasterize_edge(road_grid, edge.geometry, hw)

road_pct = 100 * (road_grid == ROAD).mean()


# make sure there are no gaps in the roads 
road_mask        = (road_grid == ROAD)
road_mask_closed = binary_closing(road_mask, structure=np.ones((3,3)))
road_grid[road_mask_closed & (road_grid == EMPTY)] = ROAD

# make sure roads are widened for visibility and egains gaps
road_mask = (road_grid == ROAD)
dist      = distance_transform_edt(~road_mask)
expanded  = dist <= EXPAND_CELLS
road_grid[expanded & (road_grid == EMPTY)] = ROAD

print(f"Rasterization complete")
print(f"Road cells : {(road_grid == ROAD).sum()}  ({road_pct:.1f}% of grid)")

### Exits and villages
Here exits are placed on random locations around the border of the network and a village is places on a random node. <br>
Exits and villages are shown larger for visibility and to make sure the algorithms do not accedentaly miss them.

In [ ]:
# find border edges to place exits on
border_candidates = (
      [(0,             c) for c in range(GRID_COLS)]
    + [(GRID_ROWS - 1, c) for c in range(GRID_COLS)]
    + [(r,             0) for r in range(GRID_ROWS)]
    + [(r, GRID_COLS - 1) for r in range(GRID_ROWS)]
)
road_border = [(r, c) for r, c in border_candidates if road_grid[r, c] == ROAD]

exits = []

if len(road_border) >= N_EXITS:
    # Use the RNG to pick N_EXITS indices without replacement
    random_indices = sim_rng.choice(len(road_border), size=N_EXITS, replace=False)
    exits = [road_border[idx] for idx in random_indices]
else:
    print(f"Warning: only {len(road_border)} border road cells found — "
          f"try a smaller CELL_SIZE_M or larger area.")
    exits = road_border[:N_EXITS] if road_border else [(0, GRID_COLS//2)]

for r, c in exits:
    for dr in range(-EXIT_RADIUS, EXIT_RADIUS + 1):
        for dc in range(-EXIT_RADIUS, EXIT_RADIUS + 1):
            if dr*dr + dc*dc <= EXIT_RADIUS**2:
                nr = np.clip(r + dr, 0, GRID_ROWS - 1)
                nc = np.clip(c + dc, 0, GRID_COLS - 1)
                if road_grid[nr, nc] in (ROAD, EMPTY):
                    road_grid[nr, nc] = EXIT

print(f"Exits placed : {len(exits)}")
print(f"Exit locations : {exits}")

global v_r, v_c, village_node
node_candidates = []
for node_id, data in G_simple.nodes(data=True):
    # Convert geographic x,y to grid r,c
    c = int((data['x'] - minx) / CELL_SIZE_M)
    r = int((maxy - data['y']) / CELL_SIZE_M)
    
    # Clip to ensure they are inside the grid
    r = np.clip(r, 0, GRID_ROWS - 1)
    c = np.clip(c, 0, GRID_COLS - 1)
    
    # Store as (row, col, node_id) so we know which node we picked later
    node_candidates.append((r, c, node_id))

if len(node_candidates) > 0:
    # Pick 1 random index
    idx = sim_rng.choice(len(node_candidates))
    v_r, v_c, village_node = node_candidates[idx]
    village_coords = [(v_r, v_c)]
    print(f"Village location: ({v_r}, {v_c})")
else:
    print("Error: No nodes found in G_simple!")


for dr in range(-VILLAGE_RADIUS, VILLAGE_RADIUS + 1):
    for dc in range(-VILLAGE_RADIUS, VILLAGE_RADIUS + 1):
         if dr*dr + dc*dc <= VILLAGE_RADIUS**2:
                nr = np.clip(v_r + dr, 0, GRID_ROWS - 1)
                nc = np.clip(v_c + dc, 0, GRID_COLS - 1)
                if road_grid[nr, nc] in (ROAD, EMPTY):
                    road_grid[nr, nc] = VILLAGE

### Visualization check
Visualize the road network to make sure everything looks as intended.

In [ ]:

def plot_grid(grid, title="Road Network", pheromone=None):
    fig, ax = plt.subplots(figsize=(10, 10))

    display = np.zeros((*grid.shape, 3))
    display[grid == EMPTY]   = [0.25, 0.40, 0.15]   # Dark green = forest
    display[grid == ROAD]    = [0.88, 0.88, 0.88]   # Light grey = road
    display[grid == EXIT]    = [0.53, 0.92, 0.28]   # Bright green = exit
    display[grid == VILLAGE]    = [0, 0., 0]        # black = village

    ax.imshow(display, interpolation='bilinear')

    if pheromone is not None:
        norm_p    = pheromone / (pheromone.max() + 1e-8)
        pher_rgba = np.zeros((*grid.shape, 4))
        pher_rgba[:, :, 2] = norm_p
        pher_rgba[:, :, 3] = norm_p * 0.75
        ax.imshow(pher_rgba, interpolation='bilinear')

    ax.set_title(title, fontsize=14, pad=10)
    ax.axis('off')

    legend_elements = [
        plt.Rectangle((0,0),1,1, color= [0.25, 0.40, 0.15], label='Forest'),
        plt.Rectangle((0,0),1,1, color=[0.88, 0.88, 0.88], label='Road'),
        plt.Rectangle((0,0),1,1, color=[0.53, 0.92, 0.28], label='Exit'),
        plt.Rectangle((0,0),1,1, color=[0, 0, 0], label='Village'),
    ]
    ax.legend(handles=legend_elements, loc='center left', 
          bbox_to_anchor=(1.02, 0.5), framealpha=0.8, fontsize=9)
    plt.tight_layout()
    plt.show()

plot_grid(road_grid)

## Wildfire layer
Here fire related functions can be found. <br>
These functions make sure the fire evolves every FIRE_SPREAD_EVERY step.<br>
-  ignite_fire places N_FIRE_STARTS amount of fire embers on random forest coordinates on the grid.
-  step_fire spreads the fire inspired by the logic of Alexandridis, A. et al.<br>
-  adjust_risk_area is an addition to the model of Alexandridis, A. et al. Here an area around the fire (of radius HIGH_RISK_RADIUS) is marked orange to show where the fire will soon spread to.


In [ ]:

BURNING          = 4
BURNED           = 5
HIGH_RISK_AREA_F = 6    # risk area on forest
HIGH_RISK_AREA_R = 7    # the risk area on roads needs to be a bit lighter


def roll_no_wrap(arr, shift, axis):
    # Make sure the fire does not roll onto the other side of the grid.
    result = np.roll(arr, shift, axis=axis)
    if axis == 0:
        if shift > 0: result[:shift, :]  = False
        else:         result[shift:, :]  = False
    elif axis == 1:
        if shift > 0: result[:, :shift]  = False
        else:         result[:, shift:]  = False
    return result

def adjust_risk_area(grid):
    # make sure the color of the risk area on the road is slightly lighter so roads are still visable. 
    risk_grid    = grid.copy()
    burning_mask = (grid == BURNING)
    y, x   = np.ogrid[-HIGH_RISK_RADIUS:HIGH_RISK_RADIUS+1,
                      -HIGH_RISK_RADIUS:HIGH_RISK_RADIUS+1]
    struct    = (x*x + y*y <= HIGH_RISK_RADIUS**2)
    risk_zone = binary_dilation(burning_mask, structure=struct)
    risk_grid[risk_zone & (grid == EMPTY)] = HIGH_RISK_AREA_F
    risk_grid[risk_zone & (grid == ROAD)]  = HIGH_RISK_AREA_R
    return risk_grid


def ignite_fire(grid, n_starts, rng):
    # pick random coordinates for the fire embers and make sure they are not along the edges
    fire_grid   = grid.copy()
    empty_cells = list(zip(*np.where(fire_grid == EMPTY)))
    margin_r    = GRID_ROWS // 10
    margin_c    = GRID_COLS // 10
    central     = [(r, c) for r, c in empty_cells
                   if margin_r < r < GRID_ROWS - margin_r
                   and margin_c < c < GRID_COLS - margin_c]
    if len(central) < n_starts:
        central = empty_cells
    chosen = rng.choice(len(central), min(n_starts, len(central)), replace=False)
    for idx in chosen:
        r, c = central[idx]
        for dr in range(r - 1, r + 2):
            for dc in range(c - 1, c + 2):
                if (0 <= dr < GRID_ROWS) and (0 <= dc < GRID_COLS):
                    fire_grid[dr, dc] = BURNING
    fire_grid = adjust_risk_area(fire_grid)
    return fire_grid


def step_fire(grid, spread_prob, rng, burn_timer):
    # Make sure the fire stays alive for BURN_DURATION steps for visibility and so it can ignight its neighbors again.
    # Roll the fire to find the new cells the fire spreads to. 
    new_grid     = grid.copy()
    new_timer    = burn_timer.copy()
    burning_mask = (grid == BURNING)
    neighbours = (
        roll_no_wrap(burning_mask,  1, axis=0) |
        roll_no_wrap(burning_mask, -1, axis=0) |
        roll_no_wrap(burning_mask,  1, axis=1) |
        roll_no_wrap(burning_mask, -1, axis=1)
    )
    spreadable  = ((grid != BURNING) & (grid != BURNED) &
                   (grid != EXIT))
    spread_roll = rng.random(grid.shape) < spread_prob
    new_fires   = neighbours & spreadable & spread_roll
    new_grid[new_fires]     = BURNING
    new_timer[new_fires]    = 0
    new_timer[burning_mask] += 1
    expires = burning_mask & (new_timer >= BURN_DURATION)
    new_grid[expires]  = BURNED
    new_timer[expires] = 0
    return new_grid, new_timer


## Physarum layer
Here Physarum related functions can be found. <br>
-  step_physarum_solver does one Physarum step according to the math of IPPA from Zhang et al.<br>
-  precompute_edge_pixels_stacked stackes all pixels for the edges for faster lookup in the future.<br>
-  get_burned_edges is used to find out which edges are either burning or burned by looking at the burned cells/pixels. <br>
-  check_path_exists finds all edges that are not burning or burned and checks if the physarum has at least one path on these edges leading from the village to one of the exits. 


In [ ]:
 
def step_physarum_solver(D, edges, lengths, b, n_nodes, burned_edges):

    # Calculate the pressure of each node 
    # According to Equation (4) and (5) in the paper:
    # sum_j [ (D_ij / L_ij) * (Pi - Pj) ] = b_i

    # We first calculate the coefficient (D_ij / L_ij) for every edge
    # Burned edges represent a physical break (D -> 0)
    D_eff = D.copy()
    D_eff[burned_edges] = 1e-6 # value to very small for math stability
    conductance = D_eff / lengths

    # Build Laplacian Matrix
    rows = np.concatenate([edges[:,0], edges[:,1], edges[:,0], edges[:,1]])
    cols = np.concatenate([edges[:,0], edges[:,1], edges[:,1], edges[:,0]])
    vals = np.concatenate([conductance, conductance, -conductance, -conductance])
    L = sp.csr_matrix((vals, (rows, cols)), shape=(n_nodes, n_nodes))

    # To solve the system, we fix one node's pressure (the sink)
    # as mentioned in the paper's implementation details.
    
    # Ground the first exit node
    anchor_idx = node_idx[exit_nodes[0]]
    L = L.tolil()
    L[anchor_idx, :] = 0
    L[anchor_idx, anchor_idx] = 1
    L = L.tocsr()
    b_fixed = b.copy()
    b_fixed[anchor_idx] = 0

    # Solve for Pressure (P)
    try:
        P = spla.spsolve(L, b_fixed)
    except:
        # If matrix is singular, return D unchanged
        return D, np.zeros(len(edges))

    # Calculate Flux (Q) 
    # According to Equation (2): Q_ij = D_ij * (Pi - Pj) / L_ij
    Q = (D_eff / lengths) * (P[edges[:, 0]] - P[edges[:, 1]])
    
    # Get p_s (village) 
    # pressure drop denominator = p_s - p_e but since p_e = 0 pressure drop is equal to p_s
    p_s = P[node_idx[village_node]]
    pressure_drop = p_s
    
    # Prevent division by zero 
    if abs(pressure_drop) < 1e-9:
        pressure_drop = 1.0
    
    # Apply the formula: Dij = 0.5 * ( (Qij * (pi - pj)) / (Lij * (ps - pe)) + Dij )
    # (P[edges[:, 0]] - P[edges[:, 1]]) is the (pi - pj) term
    numerator = Q * (P[edges[:, 0]] - P[edges[:, 1]])
    denominator = lengths * pressure_drop
    
    D_new = 0.5 * ((numerator / denominator) + D)

    return D_new, Q, L.nnz

def precompute_edge_pixels_stacked(edge_pixels):
    # Stack all edge pixel coordinates to a big array for speed.
    # Return stacked (row, col, edge_idx) array.
    
    parts = []
    for idx, pixels in enumerate(edge_pixels):
        if pixels is not None:
            edge_col = np.full((len(pixels), 1), idx, dtype=np.int32)
            parts.append(np.hstack([pixels, edge_col]))
    return np.vstack(parts) 


def get_burned_edges(stacked_pixels, sim_grid, n_edges):
    # Find all burning and burned cells and return the burning/burned edges.
    
    fire_mask = (sim_grid == BURNING) | (sim_grid == BURNED)

    rows      = stacked_pixels[:, 0]
    cols      = stacked_pixels[:, 1]
    edge_idxs = stacked_pixels[:, 2]

    # Which pixels are on fire.
    on_fire = fire_mask[rows, cols]

    # Which edges have at least one pixel on fire.
    burned  = np.zeros(n_edges, dtype=bool)
    burned[edge_idxs[on_fire]] = True

    return burned

def check_path_exists(D, edges_arr, node_idx, village_node, exit_nodes, burned_edges, threshold=1.5):
    # Look up what roads have a high-conductivity and are not on fire.
    # If there is a path from village to an exit where no part of it is on fire return True.
    functional_indices = np.where((D > threshold) & (~burned_edges))[0]
    
    if len(functional_indices) == 0:
        return False
        
    # Build a temporary graph from just the functional edges
    G_temp = nx.Graph()
    G_temp.add_edges_from(edges_arr[functional_indices])
    
    v_idx = node_idx[village_node]

    if not G_temp.has_node(v_idx):
        return False
        
    for ex in exit_nodes:
        ex_idx = node_idx[ex]
        if G_temp.has_node(v_idx) and G_temp.has_node(ex_idx):
            if nx.has_path(G_temp, v_idx, ex_idx):
                return True
    return False

## Dijkstra layer
Here Dijkstra related functions can be found. <br>
-  run_dijkstra does one Dijkstra step according to the math of Dijkstra.The build in OMNsk function multi_source_dijkstra will be used for this.<br>
-  get_burned_node_pairs converts burned edges to node pairs so Dijkstra can find them easier.  


In [ ]:
def run_dijkstra(G_simple, village_node, exit_nodes, burned_node_pairs):
    # Classic dijkstra algorithm from "A note on two problems in connexion with graphs" by Dijkstra et al.
    # Find the shortest path from the village to the nearest exit using Dijkstra but avoid burned edges.
    # Calculate 'Work' based on edges relaxed in the surviving network.

    # Make the roads burned by fire very expensive.
    def fire_weight(u, v, d):
        # Check if the edge (u, v) is in our 'burned' set.
        if (u, v) in burned_node_pairs or (v, u) in burned_node_pairs:
            return 1e9
        return d.get('weight', 1)
        
    try:
        # Find the path to nearest target and stop searcing after.
        # Iterate through targets to find the closest one
        best_dist = float('inf')
        best_path = None
        total_nodes_visited = set()

        # Find the closest exit 
        distance_map, path_map = nx.multi_source_dijkstra(
            G_simple, {village_node}, target=None, weight=fire_weight
        )
        
        # Find the specific target that was reached first (the closest exit)
        reached_targets = set(exit_nodes) & set(distance_map.keys())
        
        if not reached_targets:
            raise nx.NetworkXNoPath

        closest_target = min(reached_targets, key=lambda t: distance_map[t])
        final_path = path_map[closest_target]
        
        # Calculate 'Work': Edges relaxed are all edges connected to nodes.
        target_dist = distance_map[closest_target]
        nodes_in_search_space = [n for n, d in distance_map.items() if d <= target_dist]
        
        work = sum(G_simple.degree(n) for n in nodes_in_search_space)
        return final_path, work
        
    except (nx.NetworkXNoPath, ValueError):
        # If trapped, explore the entire reachable 'island'
        try:
            reachable_nodes = nx.descendants(G_simple, village_node) | {village_node}
            work = sum(G.degree(n) for n in reachable_nodes)
        except:
            work = 0
        return None, work


def get_burned_node_pairs(edges_arr, burned_edges, idx_to_node):
    # Convert burned edges back to (u, v) node ID pairs
    # so they can be removed from the NetworkX graph for Dijkstra.
    pairs = []
    for idx in np.where(burned_edges)[0]:
        i, j = edges_arr[idx]
        pairs.append((idx_to_node[i], idx_to_node[j]))
    return pairs


### Display functions 
These functions make sure that everything is ready to be plotted after the simulation.
-  rasterize_conductivity rasterizes all Physarum edges with a conductivity higher than threshold_draw onto a grid. <br>
-  rasterize_path rasterizes the node path of Dijkstra to a grid. <br>
-  make_display converts all current grid states to a collor and "draws" the routes of Physarum and Dijkstra on top of it. This grid will then be returned to be saved in snapshots.

In [ ]:
def rasterize_conductivity(D, edge_pixels, GRID_ROWS, GRID_COLS):
    D_grid = np.zeros((GRID_ROWS, GRID_COLS), dtype=np.float32)
 
    for idx in range(len(D)):
        # If this edge is burning/burned, don't show the coductivity.
        if burned_edges[idx]:
            continue 
            
        pixels = edge_pixels[idx]
        if pixels is not None and len(pixels) > 0:
            val = float(D[idx])
            if val > threshold_draw: 
                D_grid[pixels[:, 0], pixels[:, 1]] = np.maximum(D_grid[pixels[:, 0], pixels[:, 1]], val)
    return D_grid


def rasterize_path(path, G_simple, CELL_SIZE_M, minx, maxy, GRID_ROWS, GRID_COLS):
    # Convert the node path to a pixel grid for display.
    
    path_grid = np.zeros((GRID_ROWS, GRID_COLS), dtype=bool)
    if path is None:
        return path_grid

    nodes_proj, edges_proj_local = ox.graph_to_gdfs(G_proj)

    for k in range(len(path) - 1):
        u, v = path[k], path[k + 1]
        if not G_proj.has_edge(u, v):
            continue
        data = G_proj[u][v]
 
        if isinstance(data, dict) and 0 in data:
            data = data[0]
        if 'geometry' in data:
            coords = list(data['geometry'].coords)
        else:
            coords = [(G_proj.nodes[u]['x'], G_proj.nodes[u]['y']),
                      (G_proj.nodes[v]['x'], G_proj.nodes[v]['y'])]

        for i in range(len(coords) - 1):
            x0, y0 = coords[i]
            x1, y1 = coords[i + 1]
            n = max(2, int(np.hypot(x1-x0, y1-y0) / (CELL_SIZE_M * 0.5)))
            for t in np.linspace(0, 1, n):
                x   = x0 + t * (x1 - x0)
                y   = y0 + t * (y1 - y0)
                col = int(np.clip((x - minx) / CELL_SIZE_M, 0, GRID_COLS - 1))
                row = int(np.clip((maxy - y) / CELL_SIZE_M, 0, GRID_ROWS - 1))
                path_grid[row, col] = True

    return path_grid


def make_display(sim_grid, D_grid, dijkstra_path_grid):
    # Build an RGB display array from current simulation state.
    display_grid = adjust_risk_area(sim_grid)
    display      = np.zeros((GRID_ROWS, GRID_COLS, 3))
    display[display_grid == EMPTY]            = [0.25, 0.40, 0.15]
    display[display_grid == ROAD]             = [1, 1, 1]
    display[display_grid == EXIT]             = [0.53, 0.92, 0.28]
    display[display_grid == BURNING]          = [0.72, 0.29, 0.10]
    display[display_grid == BURNED]           = [0.24, 0.08, 0.01]
    display[display_grid == HIGH_RISK_AREA_F] = [0.92, 0.66, 0.16]
    display[display_grid == HIGH_RISK_AREA_R] = [0.91, 0.79, 0.57]


    road_mask = (sim_grid == ROAD) | (sim_grid == HIGH_RISK_AREA_R)
    
    # Physarum Layer - Blue overlay)
    if np.max(D_grid) > 0:
        v_max = np.max(D_grid)
        
        slime_mask = (D_grid > threshold_draw) & road_mask 
        
        display[slime_mask] = [0.0, 0.6, 1.0]

    #  Dijkstra path — red overlay
    if dijkstra_path_grid is not None:
        display[dijkstra_path_grid & road_mask] = [1.00, 0.00, 0.00]
        
    # Village always on top — black
    display[sim_grid == VILLAGE] = [0.00, 0.00, 0.00]

    return display


### Initialization
First some important counters will be initialized with values 0. <br>
Some tracking parameters will be initialized as well. <br>
After that, the grid will be made and the fire embers will be placed <br>
Then some nessecary work for tracking and finding nodes and indexes is done. <br>
Next the conductivity and flux for Physarum are initialized. <br>
And lastly the already burning edges are inventoried.

In [ ]:
slime_ops_to_first = 0           # This is a count to keep track of how many operations were done before the Physarum finds its first escape route.
slime_steps_to_fix_temp = 0      # This count keeps track of how many steps it takes to find a fix after all escape routes have broken.
slime_steps_to_fix_tot = 0       # This count keeps track of the total slime_steps_to_fix_temp in the whole simulation.
slime_total_ops_cumulative = 0   # This is a count that keeps track of how many operations were done during the whole simulation.

dijk_ops_to_first = 0            # This is a count to keep track of how many operations were done before the Dijkstra finds its first escape route.
dijk_steps_to_fix_temp = 0       # This count keeps track of how many steps it takes to find a fix after all escape routes have broken.
dijk_steps_to_fix_tot = 0        # This count keeps track of the total dijk_steps_to_fix_temp in the whole simulation.
dijk_total_ops_cumulative = 0    # This is a count that keeps track of how many operations were done during the whole simulation.

amount_of_fixes = 0        # This keeps track of how many times the slime mold has found a fix after the broken roads.
d_amount_of_fixes = 0      # This keeps track of how many times Dijkstra has found a fix after the broken roads.

first_route_step = None      # Step when the first path is found by Physarum
break_step = None            # Step when any active path is touched by fire
path_was_ever_found = False  # Tracks if there is an active path

d_active_path = None       # Tracks if there is an active path
d_first_route_step = None  # Step when the first path is found
d_break_step = None        # Step when any active path is touched by fire


# Build the original grid and start a fire.
sim_grid   = ignite_fire(road_grid, N_FIRE_STARTS, sim_rng)
burn_timer = np.zeros((GRID_ROWS, GRID_COLS), dtype=np.int32)



def nearest_node(r, c, G_proj, minx, maxy, CELL_SIZE_M):
    x = minx + c * CELL_SIZE_M
    y = maxy - r * CELL_SIZE_M
    return ox.distance.nearest_nodes(G_proj, x, y)

village_node = nearest_node(*village_coords[0], G_proj, minx, maxy, CELL_SIZE_M)
village_x, village_y = village_coords[0]
exit_nodes   = [nearest_node(r, c, G_proj, minx, maxy, CELL_SIZE_M)
                for r, c in exits]

largest_comp = max(nx.connected_components(G_simple), key=len)
G_simple     = G_simple.subgraph(largest_comp).copy()
print(f"Graph: {len(G_simple.nodes)} nodes, {len(G_simple.edges)} edges")

# Build node index maps 
nodes      = list(G_simple.nodes())
node_idx   = {n: i for i, n in enumerate(nodes)}
idx_to_node = {i: n for n, i in node_idx.items()}
n_nodes    = len(nodes)

village_idx = node_idx[village_node]
exit_indices = [node_idx[ex] for ex in exit_nodes if ex in node_idx]

edges_list = []
lengths_list = []
edge_pixels = []

for u, v, data in G_proj.edges(data=True):
    # Check if the nodes exist in 'node_idx'
    if u in node_idx and v in node_idx:
        edges_list.append((node_idx[u], node_idx[v]))
        lengths_list.append(max(data.get('length', 1.0), 1.0))
        
        if 'geometry' in data:
            coords = list(data['geometry'].coords)
        else:
            # If no geometry: just use node positions
            u_data = G_proj.nodes[u]
            v_data = G_proj.nodes[v]
            coords = [(u_data['x'], u_data['y']), (v_data['x'], v_data['y'])]

        all_pts = []
        for k in range(len(coords) - 1):
            x0, y0 = coords[k]
            x1, y1 = coords[k + 1]
            dist = np.hypot(x1-x0, y1-y0)
            n_samples = max(2, int(dist / (CELL_SIZE_M * 0.5)))
            for t in np.linspace(0, 1, n_samples):
                x = x0 + t * (x1 - x0)
                y = y0 + t * (y1 - y0)
                col = int(np.clip((x - minx) / CELL_SIZE_M, 0, GRID_COLS - 1))
                row = int(np.clip((maxy - y) / CELL_SIZE_M, 0, GRID_ROWS - 1))
                all_pts.append((row, col))
        
        edge_pixels.append(np.array(all_pts, dtype=np.int32))


# Convert to Final Arrays
edges_arr = np.array(edges_list)
lengths = np.array(lengths_list, dtype=np.float64)
lengths = lengths / np.mean(lengths)


# Physarum values
n_edges = len(edges_arr)
D = np.ones(n_edges, dtype=np.float64)

I0 = len(edges_arr)

# Build the b vector, this essentially keeps track of the sum of Q_ij. At the source and sink I0 and -I0 respecitvely, 0 otherwise.
b = np.zeros(n_nodes, dtype=np.float64)
b[node_idx[village_node]] = I0
for ex in exit_nodes:
    if ex in node_idx:
        b[node_idx[ex]] = -I0 / len(exit_nodes)


# Stack pixels for fast fire detection.
stacked_pixels = precompute_edge_pixels_stacked(edge_pixels)

# Check which edges are burned at the start of the simulation.
burned_edges   = get_burned_edges(stacked_pixels, sim_grid, n_edges)
burned_pairs   = get_burned_node_pairs(edges_arr, burned_edges, idx_to_node)


## Running the simulation
Here, after every FIRE_SPREAD_EVERY timestep the fire grid will be updated, while the Physarum algorithm and Dijkstra's algorithm will update every timestep. <br>
During the simulation multiple values will be tracked to calculate evaluation parameters that will be reported during or after the simulation. <br>
Whenever step is in SNAPSHOT_AT, the current state of the simulation will be saved for vizualization later on.

In [ ]:
# Run simulation and capture the snapshots
snapshots = {}
t0        = time.time()
print('\nRunning simulation...')

ACTUAL_FINAL_STEP = TOTAL_STEPS  # This parameter changes when the village is hit by fire and makes sure the simulation stops. 

village_idx = node_idx[village_node]
exit_indices = [node_idx[ex] for ex in exit_nodes if ex in node_idx]

for step in range(TOTAL_STEPS + 1):

    if step == TOTAL_STEPS + 1:
        ACTUAL_FINAL_STEP = TOTAL_STEPS + 1
        break
        
    # Fire spreads
    if step % FIRE_SPREAD_EVERY == 0:
        sim_grid, burn_timer = step_fire(sim_grid, FIRE_SPREAD_PROB, sim_rng, burn_timer)
        new_burned_indices = get_burned_edges(stacked_pixels, sim_grid, n_edges)
    
        # Increase lenghts of edges that are on fire so slime mold doesn't keep using these edges
        lengths_dynamic = lengths.copy()
        lengths_dynamic[new_burned_indices] *= 1000000
        burned_edges = new_burned_indices # Update the global burned_edges for the solver
        L_cached = None

        # Calculate how many roads are still safe
        active_road_count = np.sum(~burned_edges)
        
        # Scale I0 to prevent the slime from evaporating when the network gets small
        b[node_idx[village_node]] = 1.0 * active_road_count

        any_safe_route = check_path_exists(D, edges_arr, node_idx, village_node, exit_nodes, burned_edges, threshold=Conductivity_threshold)
        
        # If there was a path, but now there is no safe path left in the entire network all roads broke
        if path_was_ever_found and not any_safe_route and break_step is None:
            break_step = step
            print(f"Slime - Step {step}: All routes broke")

        burned_pairs   = get_burned_node_pairs(edges_arr, burned_edges, idx_to_node)

        if d_active_path is not None and d_break_step is None:

            # Convert the node list into edge pairs
            active_path_edges = list(zip(d_active_path[:-1], d_active_path[1:]))
            
            # Check if the fire has touched the current path
            path_hit = any((u, v) in burned_pairs or (v, u) in burned_pairs 
                           for u, v in active_path_edges)
            if path_hit:
                print(f"Dijkstra - Step {step}: All routes broke.")
                d_active_path = None 
                d_break_step = step

    if sim_grid[v_r, v_c] in [BURNING, BURNED]:
        print(f"Step {step}: The Village has been consumed by fire. Ending simulation.")
        ACTUAL_FINAL_STEP = step 
        break                

    # Count the amount of operations physarum has to do to find the path.
    if first_route_step is None:
        slime_ops_to_first += current_nnz

    # Count the amount of steps physarum has to do to fix a path.
    if break_step is not None:
        slime_steps_to_fix_temp += 1

    # Count the amount of operations dijkstra has to do to find the first path
    if d_first_route_step is None:
        dijk_ops_to_first += edges_relaxed

    # Count the amount of steps dijkstra has to do to fix a path
    if d_break_step is not None:
        dijk_steps_to_fix_temp += 1
        
    #Physarum step
    D, Q, current_nnz = step_physarum_solver(D, edges_arr, lengths, b, n_nodes, burned_edges)

    any_safe_route = check_path_exists(D, edges_arr, node_idx, village_node, exit_nodes, burned_edges, threshold=Conductivity_threshold)

    # Count the amount of operations physarum does every time step.
    slime_total_ops_cumulative += current_nnz        
    
    if any_safe_route:
        path_was_ever_found = True
    
    if  any_safe_route and first_route_step is None:
        # If there is a safe route and there was no route found yet, Physarum found it's first path. 
        first_route_step = step
        print(f"Slime - Step {step}: First evacuation route found")

    if break_step is not None and any_safe_route:
        # if there is a path and there was no path before, Physarum fixed the path.
        break_step = None
        amount_of_fixes += 1
        slime_steps_to_fix_tot += slime_steps_to_fix_temp
        slime_steps_to_fix_temp = 0
        print(f"Slime - Step {step}: Fix found")


    # Dijkstra step
    d_active_path, edges_relaxed  = run_dijkstra(G_simple, village_node, exit_nodes, burned_pairs)

    # Count the amount of operations dijkstra does every time step
    dijk_total_ops_cumulative += edges_relaxed
    
    if d_active_path is not None:
        # Convert the node list into edge pairs
        active_path_edges = list(zip(d_active_path[:-1], d_active_path[1:]))
        
        # Check if the fire has touched the current path
        safe_path = not any((u, v) in burned_pairs or (v, u) in burned_pairs 
                       for u, v in active_path_edges)

    # check if there is a safe escape route
    if safe_path:
        if d_first_route_step is None:
            # If there is a safe path and there was no path found yet, Dijkstra found it's first path. 
            d_first_route_step = step
            print(f"Dijkstra - Step {step}: First route found.")
            
        elif d_break_step is not None:
            # if there is a path and there was no path before, Dijkstra fixed the path.
            d_break_step = None
            d_amount_of_fixes += 1
            dijk_steps_to_fix_tot += dijk_steps_to_fix_temp
            dijk_steps_to_fix_temp = 0
            print(f"Dijkstra - Step {step}: Route recovered.")



    
    # Capture the snapshot
    if step in SNAPSHOT_AT:
        D_grid = rasterize_conductivity(D, edge_pixels, GRID_ROWS, GRID_COLS)
        dijkstra_grid = rasterize_path(d_active_path, G_simple, CELL_SIZE_M, minx, maxy, GRID_ROWS, GRID_COLS)
        snapshots[step] = make_display(sim_grid, D_grid, dijkstra_grid)
    
    

     
print(f'\nDone in {time.time()-t0:.1f}s')

# Plot table for comparison

print("\n" + "="*60)
print("                ALGORITHM EFFICIENCY")
print("="*60)
print(f"{'Phase':<25} | {'Slime Mold':<15} | {'Dijkstra':<10}")
print("-" * 60)

# Ops/Step until First Route
avg_s_first = slime_ops_to_first
avg_d_first = dijk_ops_to_first
print(f"{'Discovery (ops to First)':<25} | {avg_s_first:<15,.1f} | {avg_d_first:<10,.1f}")

# Average Ops/Step during Fix
avg_s_fix = slime_steps_to_fix_tot / amount_of_fixes if amount_of_fixes > 0 else 0
avg_d_fix = dijk_steps_to_fix_tot / d_amount_of_fixes if d_amount_of_fixes > 0 else 0
print(f"{'Rerouting (to Fix time steps)':<25} | {avg_s_fix:<15,.1f} | {avg_d_fix:<10,.1f}")

# Overall Average
global_avg_s = slime_total_ops_cumulative / (ACTUAL_FINAL_STEP)
global_avg_d = dijk_total_ops_cumulative / (ACTUAL_FINAL_STEP)
print(f"{'Overall Average (ops/step)':<25} | {global_avg_s:<15,.1f} | {global_avg_d:<10,.1f}")

print("="*60)
print("Units: Slime = Matrix Non-Zeros | Dijkstra = Edges Relaxed")

## Visualizing the outcomes 
Here the safed snapshots will be plotted, each element having its one colour. The legend provides an explanation to what all colours mean.

In [ ]:

# Plot the snapshots 
n     = len(snapshots)
ncols = 3
nrows = int(np.ceil((n + 1) / ncols))   # +1 for legend

fig, axes = plt.subplots(nrows, ncols,
                          figsize=(ncols * 5, nrows * 5))
axes = np.array(axes).flatten()

legend_elements = [
    mpatches.Patch(color=[0.25, 0.40, 0.15], label='Forest'),
    mpatches.Patch(color=[1, 1, 1], label='Road'),
    mpatches.Patch(color=[0.53, 0.92, 0.28], label='Exit'),
    mpatches.Patch(color=[0.00, 0.00, 0.00], label='Village'),
    mpatches.Patch(color=[0.72, 0.29, 0.10], label='Fire'),
    mpatches.Patch(color=[0.24, 0.08, 0.01], label='Burned'),
    mpatches.Patch(color=[0.92, 0.66, 0.16], label='Risk area'),
    mpatches.Patch(color=[0.00, 0.60, 1.00], label='Physarum route'),
    mpatches.Patch(color=[1.00, 0.00, 0.00], label='Dijkstra route'),
]

for i, (step, img) in enumerate(sorted(snapshots.items())):
    axes[i].imshow(img, interpolation='nearest')
    axes[i].set_title(f'Step {step}', fontsize=12)
    axes[i].axis('off')

# Legend
axes[n].legend(handles=legend_elements, loc='center',
               fontsize=10, framealpha=0.9)
axes[n].axis('off')

# Hide unused panels
for j in range(n + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle('Slime Mold vs. Dijkstra Wildfire Evacuation',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('test_snapshots.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved as test_snapshots.png ')
